[개선 프로젝트 주제] "생물학적 계층 구조 및 유전적 유사성(KIPAN-KIRC 등)을 활용한 26개 암종 멀티클래스 분류 모델 성능 개선"

배경 및 한계점: 4,384개의 고차원 유전자 변이 데이터와 극희소(Sparse, ~0.81%) 특성을 가진 26개 암종 데이터 학습 시, 단순 플랫(Flat) 분류 방식은 클래스 간 경계 모호성과 과적합으로 인해 성능 한계에 직면함.   

핵심 전처리 및 모델링 전략
1. 유전적 계층화 및 그룹화: KIPAN-KIRC 관계와 같이 생물학적/통계적으로 유사한 암종군 간의 공통 변이 패턴을 선행 학습.
2. 계층적/멀티태스크 구조 도입: 상위 암종 그룹 예측과 26개 세부 암종 예측을 연계하여 데이터 희소성이 높은 클래스의 학습 보완.
3. 희소 데이터 최적화: 0.81%의 낮은 Mutation Ratio 속에서 유의미한 패턴을 추출할 수 있도록 피처 스페이스 재구성.   
4. 기대 효과: 유사 암종 간 오분류(False Positive/Negative) 감소 및 소수 클래스(예: DLBC 등)에 대한 일반화 성능 향상.

[Raw Data: 4,384 Genes, 6,201 Samples, 26 Subclasses]
       ↓
Step 1: 메타 암종(예: KIPAN)과 세부 암종(예: KIRC) 간 계층 레이블(Hierarchical Label) 매핑
       ↓
Step 2: 유전자 변이 프로필 기반 유사도 행렬 생성 및 암종별 클러스터링(Group)
       ↓
Step 3: 희소성(0.81%) 대응 및 유전자 피처 가공 (차원 축소 / 공발생 패턴 피처)
       ↓
Step 4: 계층적 멀티태스크(Multi-task) 학습용 PyTorch Dataset & Model 구성

In [1]:
import numpy    as np
import pandas   as pd
import re

from scipy.sparse       import csr_matrix
from sklearn.cluster    import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset


# =============================================================
# [Step 1] 데이터 로드 및 변이 심각도 점수화 (0~3 점수 스코어링)
# =============================================================

# 1. 데이터 로드
train = pd.read_csv("./sample/train.csv")
test = pd.read_csv("./sample/test.csv")

print(f"Train 데이터 크기: {train.shape}")
print(f"Test 데이터 크기: {test.shape}")

# 2. 유전자 피처 컬럼 추출
exclude_cols = [
    'ID',
    'SUBCLASS',
    'SUPER_CLASS',
    'sub_label',
    'super_label'
]

gene_cols = [
    col for col in train.columns
    if col not in exclude_cols
]

# Test 컬럼 정합성 확인
assert set(gene_cols).issubset(set(test.columns)), \
    "Test 데이터에 Train 유전자 컬럼이 존재하지 않습니다."

print(f"추출된 유전자 피처 개수: {len(gene_cols)}개")


# 3. 변이 심각도 변환 함수 정의
NO_CHANGE = re.compile(r'[A-Z]\d+')  # 위치만 있고 변화 표기가 없는 형태 (예: M364)

def encode_token(t: str) -> int:
    """변이 하나를 0~3 점수로 변환. 해석 못하면 -1"""
    if 'fs' in t:                                        # frameshift
        return 3
    if '>' in t:                                         # 치환/복합 변이
        left, right = t.split('>')
        left = re.sub(r'^[\d_\-]+', '', left)
        if right.endswith('*'):
            return 3
        return 1 if left == right else 2
    if re.fullmatch(r'[A-Z]\d+\*', t):                   # nonsense
        return 3
    if re.fullmatch(r'\*\d+\*', t):                      # 종결코돈 유지
        return 1
    if 'del' in t or 'ins' in t:                         # in-frame 결실/삽입
        return 3 if t.endswith('*') else 2
    m = re.fullmatch(r'([A-Z])\d+([A-Z])', t)            # 단일 아미노산 치환
    if m:
        return 1 if m.group(1) == m.group(2) else 2
    if NO_CHANGE.fullmatch(t):                           # 위치만 있는 변이
        return 2
    return -1


def encode_cell(v: str) -> int:
    if v == 'WT' or pd.isna(v):
        return 0
    return max(encode_token(t) for t in str(v).split())  # 한 칸에 변이가 여러 개면 최댓값


def unknown_tokens(values):
    return sorted({t for v in values if v != 'WT' and pd.notna(v) for t in str(v).split() if encode_token(t) == -1})


def encode_frame(df_raw):
    """문자열 변이 표 -> int8 점수 표. 고유 문자열만 변환해서 빠르게 처리"""
    df_raw = df_raw.fillna('WT')
    uniq = pd.unique(df_raw.values.ravel())
    bad = unknown_tokens(uniq)
    assert not bad, f"해석 못한 변이 표기: {bad[:20]}"

    mp = {v: encode_cell(v) for v in uniq}
    
    # 각 컬럼에 mapping 적용
    encoded_df = df_raw[gene_cols].apply(lambda col: col.map(mp)).astype(np.int8)
    return encoded_df

Train 데이터 크기: (6201, 4386)
Test 데이터 크기: (2546, 4385)
추출된 유전자 피처 개수: 4384개


In [2]:
# -------------------------------------------------------------
# [Step 2] 계층적 레이블(Super Class) 매핑 정의 (연구 질문 1, 2 대응)
# -------------------------------------------------------------
# 예: KIPAN 내부에 KIRC, KIRP 등이 포함되는 구조나, 유사한 장기/계열별 그룹 정의
def get_super_class_mapping(subclass):

    # 1. 신장암 계열
    if subclass in ["KIRC", "KIRP", "KICH", "KIPAN"]:
        return "Kidney_Group"

    # 2. 폐암 계열
    elif subclass in ["LUAD", "LUSC"]:
        return "Lung_Group"

    # 3. 소화기 계열
    elif subclass in ["STES", "COAD", "LIHC", "PAAD"]:
        return "GI_Digestive_Group"

    # 4. 뇌 및 신경계
    elif subclass in ["GBMLGG", "LGG"]:
        return "CNS_Brain_Group"

    # 5. 여성 생식기 계열
    elif subclass in ["BRCA", "CESC", "UCEC", "OV"]:
        return "Female_Reproductive_Group"

    # 6. 남성 생식기 및 비뇨기 계열
    elif subclass in ["PRAD", "BLCA", "TGCT"]:
        return "Male_Genitourinary_Group"

    # 7. 두경부 계열
    elif subclass in ["HNSC"]:
        return "Head_Neck_Group"

    # 8. 내분비 및 희귀 암종
    elif subclass in ["THCA", "THYM", "ACC", "PCPG"]:
        return "Endocrine_Rare_Group"

    # 9. 피부 및 육종
    elif subclass in ["SKCM", "SARC"]:
        return "Skin_Sarcoma_Group"

    # 10. 조혈 및 림프계
    elif subclass in ["LAML", "DLBC"]:
        return "Hematopoietic_Group"

    else:
        return "Other_Group"


# 상위 그룹 레이블 생성
train["SUPER_CLASS"] = (
    train["SUBCLASS"]
    .apply(get_super_class_mapping)
)

# Label Encoding
sub_encoder = LabelEncoder()
super_encoder = LabelEncoder()

train["sub_label"] = (
    sub_encoder.fit_transform(train["SUBCLASS"])
)

train["super_label"] = (
    super_encoder.fit_transform(train["SUPER_CLASS"])
)

num_sub_classes = len(sub_encoder.classes_)
num_super_classes = len(super_encoder.classes_)

print("\n[계층적 레이블 생성 결과]")
print(f"세부 암종(Subclass): {num_sub_classes}개")
print(f"상위 그룹(Super Class): {num_super_classes}개")

print("\n[상위 그룹별 암종 분포]")
print(
    train.groupby("SUPER_CLASS")["SUBCLASS"]
    .value_counts()
)


[계층적 레이블 생성 결과]
세부 암종(Subclass): 26개
상위 그룹(Super Class): 10개

[상위 그룹별 암종 분포]
SUPER_CLASS                SUBCLASS
CNS_Brain_Group            GBMLGG      461
                           LGG         229
Endocrine_Rare_Group       THCA        324
                           PCPG        147
                           THYM         98
                           ACC          72
Female_Reproductive_Group  BRCA        786
                           OV          253
                           UCEC        198
                           CESC        155
GI_Digestive_Group         STES        379
                           COAD        223
                           LIHC        158
                           PAAD        120
Head_Neck_Group            HNSC        223
Hematopoietic_Group        LAML        158
                           DLBC         38
Kidney_Group               KIPAN       515
                           KIRC        334
Lung_Group                 LUAD        184
                          

In [3]:
# =============================================================
# [Step 3] 데이터 분리 및 누수 없는 피처 정제 (0~3점 스코어링 적용)
# =============================================================

print("\n고차원 유전자 변이 데이터 점수화 및 정제 시작...")

# 1. train, test의 유전자 컬럼에 0~3점 변이 심각도 스코어링 적용
X_train_raw = encode_frame(train[gene_cols])
X_test_raw = encode_frame(test[gene_cols])

# 2. 점수화된 값이 0, 1, 2, 3 범위 내에 있는지 확인
assert X_train_raw.isin([0, 1, 2, 3]).all().all()
assert X_test_raw.isin([0, 1, 2, 3]).all().all()

y_sub = train["sub_label"]
y_super = train["super_label"]

# 3. Train / Validation 분리
train_idx, val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=42,
    stratify=y_sub
)

X_train_split = X_train_raw.iloc[train_idx].copy()
X_val_split = X_train_raw.iloc[val_idx].copy()

y_sub_train = y_sub.iloc[train_idx].copy()
y_sub_val = y_sub.iloc[val_idx].copy()

y_super_train = y_super.iloc[train_idx].copy()
y_super_val = y_super.iloc[val_idx].copy()

# 4. 학습 데이터만으로 Active Gene 선택 (변이 점수 합이 0보다 큰 유전자)
active_genes = X_train_split.columns[
    X_train_split.sum(axis=0) > 0
]

X_train_filtered = X_train_split[active_genes].copy()
X_val_filtered = X_val_split[active_genes].copy()
X_test_filtered = X_test_raw[active_genes].copy()

assert list(X_train_filtered.columns) == list(X_val_filtered.columns)
assert list(X_train_filtered.columns) == list(X_test_filtered.columns)

X_train_gene = X_train_filtered.to_numpy(dtype=np.float32)
X_val_gene = X_val_filtered.to_numpy(dtype=np.float32)
X_test_gene = X_test_filtered.to_numpy(dtype=np.float32)

print("\n모델 입력 데이터 변환 완료!")
print(f"Train gene shape: {X_train_gene.shape}, Val gene shape: {X_val_gene.shape}")


고차원 유전자 변이 데이터 점수화 및 정제 시작...

모델 입력 데이터 변환 완료!
Train gene shape: (4960, 4230), Val gene shape: (1241, 4230)


In [4]:
def analyze_subclass_similarity(
    X_data,
    y_subclass,
    n_clusters=5
):

    # 1. 입력 데이터 DataFrame 변환
    temp_df = pd.DataFrame(X_data)

    # 2. SUBCLASS 레이블 결합
    subclass_series = pd.Series(
        np.asarray(y_subclass),
        index=temp_df.index,
        name="SUBCLASS"
    )

    temp_df["SUBCLASS"] = subclass_series

    # 3. 암종별 평균 유전자 변이 패턴 계산
    centroids = (
        temp_df
        .groupby("SUBCLASS")
        .mean()
    )

    # 4. 클러스터 개수 유효성 확인
    if n_clusters < 2 or n_clusters > len(centroids):
        raise ValueError(
            "n_clusters는 2 이상, 암종 개수 이하이어야 합니다."
        )

    # 5. 계층적 클러스터링
    clustering = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="euclidean",
        linkage="ward"
    )

    cluster_ids = clustering.fit_predict(centroids)

    # 6. 결과 정리
    similarity_report = pd.DataFrame({
        "Subclass": centroids.index,
        "Cluster_ID": cluster_ids
    })

    # 7. 암종별 클러스터 결과 반환
    return similarity_report, centroids


# Train 데이터만 사용하여 유사성 분석
# Validation 데이터는 클러스터링 과정에 사용하지 않음
similarity_df, subclass_centroids = (
    analyze_subclass_similarity(
        X_train_gene,
        train["SUBCLASS"].iloc[train_idx],
        n_clusters=5
    )
)

print("\n[암종별 유전적 유사성 클러스터링 결과]")
print(
    similarity_df.sort_values("Cluster_ID")
    .to_string(index=False)
)

print("\n[클러스터별 암종 구성]")
for cluster_id, group in similarity_df.groupby("Cluster_ID"):

    print(
        f"\nCluster {cluster_id}: "
        f"{group['Subclass'].tolist()}"
    )

print("\n[클러스터링 분석 완료]")


[암종별 유전적 유사성 클러스터링 결과]
Subclass  Cluster_ID
     ACC           0
    THCA           0
    TGCT           0
    SARC           0
    PRAD           0
    PCPG           0
    PAAD           0
      OV           0
    THYM           0
     LGG           0
    LIHC           0
    KIRC           0
   KIPAN           0
    HNSC           0
  GBMLGG           0
    CESC           0
    BRCA           0
    LAML           0
    LUAD           1
    LUSC           1
    COAD           1
    STES           1
    BLCA           1
    UCEC           2
    SKCM           3
    DLBC           4

[클러스터별 암종 구성]

Cluster 0: ['ACC', 'BRCA', 'CESC', 'GBMLGG', 'HNSC', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LIHC', 'OV', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'TGCT', 'THCA', 'THYM']

Cluster 1: ['BLCA', 'COAD', 'LUAD', 'LUSC', 'STES']

Cluster 2: ['UCEC']

Cluster 3: ['SKCM']

Cluster 4: ['DLBC']

[클러스터링 분석 완료]


In [5]:
# =============================================================
# [Step 5-1] PyTorch Dataset 구성
# =============================================================

class CancerDataset(Dataset):

    def __init__(self, features, super_labels, sub_labels):

        self.features = torch.tensor(
            features, dtype=torch.float32
        )

        self.super_labels = torch.tensor(
            np.asarray(super_labels),
            dtype=torch.long
        )

        self.sub_labels = torch.tensor(
            np.asarray(sub_labels),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):

        return (
            self.features[idx],
            self.super_labels[idx],
            self.sub_labels[idx]
        )


# Step 3에서 이미 분리한 데이터를 그대로 사용
train_dataset = CancerDataset(
    X_train_gene,
    y_super_train,
    y_sub_train
)

val_dataset = CancerDataset(
    X_val_gene,
    y_super_val,
    y_sub_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

print("[Dataset 구성 완료]")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

[Dataset 구성 완료]
Train samples: 4960
Validation samples: 1241


In [6]:
# =============================================================
# [Step 5-2] 디바이스 및 클래스 가중치 설정
# =============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

input_dim = X_train_gene.shape[1]

# Train 데이터만 사용하여 클래스 가중치 계산
sub_classes_array = np.asarray(y_sub_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_sub_classes),
    y=sub_classes_array
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(f"\n사용 디바이스: {device}")
print(f"Input dimension: {input_dim}")
print(f"Subclass classes: {num_sub_classes}")
print(f"Super classes: {num_super_classes}")


사용 디바이스: cpu
Input dimension: 4230
Subclass classes: 26
Super classes: 10


In [7]:
# =============================================================
# [Step 5-3] 계층적 멀티태스크 분류 모델
# =============================================================

class SoftHierarchicalCancerClassifier(nn.Module):
    def __init__(self, input_dim, num_super_classes, num_sub_classes, sub_to_super_mapping, hidden_dim=256):
        """
        sub_to_super_mapping: 각 sub_label이 몇 번째 super_label에 속하는지 나타내는 1D Tensor/Array (크기: 26)
        """
        super().__init__()
        self.num_super_classes = num_super_classes
        self.num_sub_classes = num_sub_classes
        
        # sub_label -> super_label 매핑 텐서 등록 (버퍼로 저장)
        self.register_buffer("sub_to_super", torch.tensor(sub_to_super_mapping, dtype=torch.long))

        # 1. 공유 특징 추출기 (Shared Encoder)
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # 2. 상위 그룹 분류기 (Super-Class Gate/Router)
        self.super_classifier = nn.Linear(hidden_dim // 2, num_super_classes)

        # 3. 세부 암종 분류기 (Sub-Class Classifier)
        self.sub_classifier = nn.Linear(hidden_dim // 2, num_sub_classes)

    def forward(self, x):
        features = self.shared_encoder(x)
        
        # 상위 그룹 및 세부 암종 Logits 계산
        super_logits = self.super_classifier(features)
        sub_logits = self.sub_classifier(features)
        
        # Softmax를 적용하여 각각 확률 분포로 변환
        P_super = F.softmax(super_logits, dim=1) # Shape: [Batch, 10]
        P_sub_cond = F.softmax(sub_logits, dim=1) # Shape: [Batch, 26]
        
        # 4. 생물학적 계층 결합: P(Sub_c) = P(Super_g(c)) * P(Sub_c | Super_g(c))
        # 각 sub_class c에 대응하는 super_class 확률을 선택해서 곱함
        P_super_g = P_super[:, self.sub_to_super] # Shape: [Batch, 26]
        
        # 최종 통합 세부 암종 확률 (계층 조건부 확률 반영)
        P_sub_final = P_sub_cond * P_super_g
        
        return super_logits, sub_logits, P_sub_final

# 세부 암종(0~25)이 속한 상위 그룹(0~9) 매핑 배열 생성
sub_to_super_arr = (
    train.groupby("sub_label")["super_label"].first()
    .reindex(range(num_sub_classes)).to_numpy()
)

# 모델 생성
model = SoftHierarchicalCancerClassifier(
    input_dim=input_dim,
    num_super_classes=num_super_classes,
    num_sub_classes=num_sub_classes,
    sub_to_super_mapping=sub_to_super_arr,
    hidden_dim=256
).to(device)

In [8]:
# =============================================================
# [Step 5-4] Loss 및 Optimizer 설정
# =============================================================

# 손실 함수 설정
criterion_super = nn.CrossEntropyLoss()
criterion_sub = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

alpha = 0.6  # 상위 그룹 loss의 비중

print("\n모델 및 손실 함수 설정 완료!")




모델 및 손실 함수 설정 완료!


In [9]:
# =============================================================
# [Step 5-5] 멀티태스크 학습
# =============================================================

num_epochs = 50

history = {
    "train_loss": [],
    "val_loss": [],
    "val_sub_acc": [],
    "val_sub_f1": [],
    "val_super_acc": []
}

best_val_f1 = -1.0
best_model_state = None

print("\n[모델 학습 시작]")

for epoch in range(num_epochs):

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0
    
    for features, super_targets, sub_targets in train_loader:
        features = features.to(device)
        super_targets = super_targets.to(device)
        sub_targets = sub_targets.to(device)

        optimizer.zero_grad()
        
        super_logits, sub_logits, P_sub_final = model(features)
        
        # 1. 상위 그룹 손실
        loss_super = criterion_super(super_logits, super_targets)
        # 2. 세부 암종 손실 (Direct Logits Loss)
        loss_sub_direct = criterion_sub(sub_logits, sub_targets)
        # 3. 계층 확률 통합 NLL Loss (안정적인 최적화를 위해 log 변환)
        loss_sub_hier = F.nll_loss(torch.log(P_sub_final + 1e-8), sub_targets, weight=class_weights_tensor)
        
        # 총 손실 = 세부 암종 손실 + 계층 통합 손실 + 상위 그룹 가이딩 손실
        total_loss = loss_sub_direct + loss_sub_hier + (alpha * loss_super)

        total_loss.backward()
        optimizer.step()

        batch_size = features.size(0)

        train_loss_sum += total_loss.item() * batch_size
        train_sample_count += batch_size

    train_loss = train_loss_sum / train_sample_count

# ---------------------------------------------------------
    # Validation
    # ---------------------------------------------------------

    model.eval()

    val_loss_sum = 0.0
    val_sample_count = 0

    val_sub_preds = []
    val_sub_targets = []

    val_super_preds = []
    val_super_targets = []

    with torch.no_grad():

        for features, super_targets, sub_targets in val_loader:

            features = features.to(device)
            super_targets = super_targets.to(device)
            sub_targets = sub_targets.to(device)

            # 💡 수정: 모델이 반환하는 3개의 값(상위 logits, 세부 logits, 계층 통합 확률)을 모두 받음
            super_logits, sub_logits, P_sub_final = model(features)

            loss_super = criterion_super(
                super_logits,
                super_targets
            )

            # 💡 수정: 학습 시와 동일하게 세부 직접 손실 및 계층 통합 NLL Loss 반영
            loss_sub_direct = criterion_sub(
                sub_logits,
                sub_targets
            )
            
            loss_sub_hier = F.nll_loss(
                torch.log(P_sub_final + 1e-8), 
                sub_targets, 
                weight=class_weights_tensor
            )

            total_loss = loss_sub_direct + loss_sub_hier + (alpha * loss_super)

            batch_size = features.size(0)

            val_loss_sum += total_loss.item() * batch_size
            val_sample_count += batch_size

            # 💡 수정: 세부 암종 예측은 계층 구조의 확률이 반영된 P_sub_final을 기준으로 argmax 수행
            sub_preds = torch.argmax(
                P_sub_final,
                dim=1
            )

            super_preds = torch.argmax(
                super_logits,
                dim=1
            )

            val_sub_preds.extend(
                sub_preds.cpu().numpy()
            )

            val_sub_targets.extend(
                sub_targets.cpu().numpy()
            )

            val_super_preds.extend(
                super_preds.cpu().numpy()
            )

            val_super_targets.extend(
                super_targets.cpu().numpy()
            )

    val_loss = val_loss_sum / val_sample_count

    # Subclass 성능
    val_sub_acc = accuracy_score(
        val_sub_targets,
        val_sub_preds
    )

    val_sub_f1 = f1_score(
        val_sub_targets,
        val_sub_preds,
        average="macro",
        zero_division=0
    )

    # Super Class 성능
    val_super_acc = accuracy_score(
        val_super_targets,
        val_super_preds
    )

    # 학습 기록
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_sub_acc"].append(val_sub_acc)
    history["val_sub_f1"].append(val_sub_f1)
    history["val_super_acc"].append(val_super_acc)

    # Macro F1 기준 최적 모델 저장
    if val_sub_f1 > best_val_f1:

        best_val_f1 = val_sub_f1

        best_model_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Sub Acc: {val_sub_acc:.4f} "
        f"| Sub Macro F1: {val_sub_f1:.4f} "
        f"| Super Acc: {val_super_acc:.4f}"
    )


[모델 학습 시작]
Epoch [01/50] | Train Loss: 9.0260 | Val Loss: 7.4135 | Sub Acc: 0.3183 | Sub Macro F1: 0.2536 | Super Acc: 0.5077
Epoch [02/50] | Train Loss: 2.9411 | Val Loss: 6.5531 | Sub Acc: 0.3400 | Sub Macro F1: 0.3034 | Super Acc: 0.5592
Epoch [03/50] | Train Loss: 1.1217 | Val Loss: 6.9877 | Sub Acc: 0.3320 | Sub Macro F1: 0.3054 | Super Acc: 0.5439
Epoch [04/50] | Train Loss: 0.4623 | Val Loss: 7.7357 | Sub Acc: 0.3175 | Sub Macro F1: 0.3008 | Super Acc: 0.5391
Epoch [05/50] | Train Loss: 0.2332 | Val Loss: 8.4262 | Sub Acc: 0.3151 | Sub Macro F1: 0.3018 | Super Acc: 0.5375
Epoch [06/50] | Train Loss: 0.1372 | Val Loss: 8.9189 | Sub Acc: 0.3110 | Sub Macro F1: 0.2980 | Super Acc: 0.5318
Epoch [07/50] | Train Loss: 0.0934 | Val Loss: 9.4948 | Sub Acc: 0.3102 | Sub Macro F1: 0.2967 | Super Acc: 0.5383
Epoch [08/50] | Train Loss: 0.0711 | Val Loss: 9.7976 | Sub Acc: 0.2957 | Sub Macro F1: 0.2816 | Super Acc: 0.5238
Epoch [09/50] | Train Loss: 0.0560 | Val Loss: 10.1538 | Sub Acc: 0.

In [10]:
# =============================================================
# [Step 5-6] 최적 모델 복원
# =============================================================

if best_model_state is None:
    raise RuntimeError("학습 중 최적 모델이 저장되지 않았습니다.")

model.load_state_dict(best_model_state)
model.to(device)

print("\n[학습 완료]")
print(f"Best Validation Subclass Macro F1: {best_val_f1:.4f}")


[학습 완료]
Best Validation Subclass Macro F1: 0.3054


In [11]:
# =============================================================
# [Step 6-1] Validation 예측 수행
# =============================================================

model.eval()

val_preds_sub = []
val_preds_super = []

val_true_sub = []
val_true_super = []

with torch.no_grad():

    for features, super_targets, sub_targets in val_loader:

        features = features.to(device)

        super_logits, sub_logits, P_sub_final = model(features)

        preds_sub = torch.argmax(sub_logits, dim=1)
        preds_super = torch.argmax(super_logits, dim=1)

        val_preds_sub.extend(
            preds_sub.cpu().numpy()
        )

        val_preds_super.extend(
            preds_super.cpu().numpy()
        )

        val_true_sub.extend(
            sub_targets.numpy()
        )

        val_true_super.extend(
            super_targets.numpy()
        )

In [12]:
# =============================================================
# [Step 6-2] 성능 평가
# =============================================================

# Subclass 평가
sub_accuracy = accuracy_score(
    val_true_sub,
    val_preds_sub
)

sub_macro_f1 = f1_score(
    val_true_sub,
    val_preds_sub,
    average="macro",
    zero_division=0
)

sub_weighted_f1 = f1_score(
    val_true_sub,
    val_preds_sub,
    average="weighted",
    zero_division=0
)

# Super Class 평가
super_accuracy = accuracy_score(
    val_true_super,
    val_preds_super
)

super_macro_f1 = f1_score(
    val_true_super,
    val_preds_super,
    average="macro",
    zero_division=0
)

print("\n========== Validation 성능 평가 ==========")

print(
    f"Subclass Accuracy: {sub_accuracy:.4f}"
)

print(
    f"Subclass Macro F1: {sub_macro_f1:.4f}"
)

print(
    f"Subclass Weighted F1: {sub_weighted_f1:.4f}"
)

print(
    f"Super Class Accuracy: {super_accuracy:.4f}"
)

print(
    f"Super Class Macro F1: {super_macro_f1:.4f}"
)


========== Validation 성능 평가 ==========
Subclass Accuracy: 0.3344
Subclass Macro F1: 0.3117
Subclass Weighted F1: 0.3244
Super Class Accuracy: 0.5439
Super Class Macro F1: 0.4883


In [13]:
# =============================================================
# [Step 6-3] 예측 결과 DataFrame 생성
# =============================================================

val_results_df = pd.DataFrame({

    "true_sub_label": val_true_sub,
    "pred_sub_label": val_preds_sub,

    "true_super_label": val_true_super,
    "pred_super_label": val_preds_super

})

# 원래 암종 이름 복원
val_results_df["true_sub_name"] = (
    sub_encoder.inverse_transform(
        val_results_df["true_sub_label"]
    )
)

val_results_df["pred_sub_name"] = (
    sub_encoder.inverse_transform(
        val_results_df["pred_sub_label"]
    )
)

# 상위 그룹 이름 복원
val_results_df["true_super_name"] = (
    super_encoder.inverse_transform(
        val_results_df["true_super_label"]
    )
)

val_results_df["pred_super_name"] = (
    super_encoder.inverse_transform(
        val_results_df["pred_super_label"]
    )
)

# 오분류 여부
val_results_df["is_error"] = (
    val_results_df["true_sub_label"]
    != val_results_df["pred_sub_label"]
)

# 상위 그룹 오분류 여부
val_results_df["is_super_error"] = (
    val_results_df["true_super_label"]
    != val_results_df["pred_super_label"]
)

print("\n[Validation 예측 결과 미리보기]")
print(val_results_df.head())



[Validation 예측 결과 미리보기]
   true_sub_label  pred_sub_label  true_super_label  pred_super_label  \
0              22              23                 8                 1   
1               8              22                 6                 8   
2              21              14                 3                 7   
3              25              25                 2                 2   
4              14              20                 7                 7   

  true_sub_name pred_sub_name            true_super_name  \
0          TGCT          THCA   Male_Genitourinary_Group   
1         KIPAN          TGCT               Kidney_Group   
2          STES          LUSC         GI_Digestive_Group   
3          UCEC          UCEC  Female_Reproductive_Group   
4          LUSC          SKCM                 Lung_Group   

             pred_super_name  is_error  is_super_error  
0       Endocrine_Rare_Group      True            True  
1   Male_Genitourinary_Group      True            True  
2   

In [14]:
# =============================================================
# [Step 6-4] 암종별 Classification Report
# =============================================================

print("\n========== Subclass Classification Report ==========")

print(
    classification_report(
        val_true_sub,
        val_preds_sub,
        labels=np.arange(num_sub_classes),
        target_names=sub_encoder.classes_,
        zero_division=0
    )
)

print("\n========== Super Class Classification Report ==========")

print(
    classification_report(
        val_true_super,
        val_preds_super,
        labels=np.arange(num_super_classes),
        target_names=super_encoder.classes_,
        zero_division=0
    )
)


========== Subclass Classification Report ==========
              precision    recall  f1-score   support

         ACC       0.43      0.71      0.54        14
        BLCA       0.17      0.19      0.18        21
        BRCA       0.58      0.38      0.46       157
        CESC       0.07      0.10      0.08        31
        COAD       0.61      0.60      0.61        45
        DLBC       0.27      0.43      0.33         7
      GBMLGG       0.37      0.17      0.24        92
        HNSC       0.23      0.20      0.21        45
       KIPAN       0.14      0.10      0.11       103
        KIRC       0.17      0.21      0.19        67
        LAML       0.43      0.56      0.49        32
         LGG       0.41      0.78      0.54        46
        LIHC       0.28      0.23      0.25        31
        LUAD       0.21      0.16      0.18        37
        LUSC       0.14      0.14      0.14        36
          OV       0.22      0.27      0.24        51
        PAAD       0.17    

In [15]:
# =============================================================
# [Step 6-5] 오분류 환자군 추출
# =============================================================

error_samples_df = val_results_df[
    val_results_df["is_error"]
].copy()

total_samples = len(val_results_df)
error_count = len(error_samples_df)

error_rate = (
    error_count / total_samples * 100
)

print("\n========== 오분류 분석 ==========")

print(f"Validation 전체 샘플: {total_samples}")
print(f"오분류 샘플: {error_count}")
print(f"오분류율: {error_rate:.2f}%")



========== 오분류 분석 ==========
Validation 전체 샘플: 1241
오분류 샘플: 826
오분류율: 66.56%


In [16]:
# =============================================================
# [Step 6-6] 암종별 오분류율 분석
# =============================================================

subclass_error_analysis = (
    val_results_df
    .groupby("true_sub_name")
    .agg(
        Total_Count=("is_error", "size"),
        Error_Count=("is_error", "sum")
    )
)

subclass_error_analysis["Error_Rate"] = (
    subclass_error_analysis["Error_Count"]
    / subclass_error_analysis["Total_Count"]
)

subclass_error_analysis["Accuracy"] = (
    1 - subclass_error_analysis["Error_Rate"]
)

subclass_error_analysis = (
    subclass_error_analysis
    .sort_values("Error_Rate", ascending=False)
)

print("\n[암종별 오분류율]")
print(
    subclass_error_analysis.to_string()
)



[암종별 오분류율]
               Total_Count  Error_Count  Error_Rate  Accuracy
true_sub_name                                                
SARC                    40           37    0.925000  0.075000
CESC                    31           28    0.903226  0.096774
KIPAN                  103           93    0.902913  0.097087
PCPG                    29           26    0.896552  0.103448
LUSC                    36           31    0.861111  0.138889
LUAD                    37           31    0.837838  0.162162
GBMLGG                  92           76    0.826087  0.173913
BLCA                    21           17    0.809524  0.190476
HNSC                    45           36    0.800000  0.200000
KIRC                    67           53    0.791045  0.208955
LIHC                    31           24    0.774194  0.225806
PRAD                    53           41    0.773585  0.226415
PAAD                    24           18    0.750000  0.250000
OV                      51           37    0.725490  0.274

In [17]:
# =============================================================
# [Step 6-7] 암종 간 오분류 패턴 분석
# =============================================================

confusion_pairs = (
    error_samples_df
    .groupby(
        ["true_sub_name", "pred_sub_name"]
    )
    .size()
    .reset_index(name="confusion_count")
    .sort_values(
        "confusion_count",
        ascending=False
    )
)

print("\n[가장 빈번한 암종 간 오분류 패턴 Top 10]")

print(
    confusion_pairs.head(10).to_string(index=False)
)


[가장 빈번한 암종 간 오분류 패턴 Top 10]
true_sub_name pred_sub_name  confusion_count
        KIPAN          KIRC               58
       GBMLGG           LGG               45
         KIRC         KIPAN               45
         BRCA            OV               20
         THCA          THYM               11
         BRCA          UCEC               11
         COAD          STES                9
         STES          HNSC                9
         STES          LUSC                9
         PCPG          THYM                8


In [18]:
# =============================================================
# [Step 6-8] 상위 그룹 간 오분류 분석
# =============================================================

super_error_pairs = (
    val_results_df[
        val_results_df["is_super_error"]
    ]
    .groupby(
        ["true_super_name", "pred_super_name"]
    )
    .size()
    .reset_index(name="confusion_count")
    .sort_values(
        "confusion_count",
        ascending=False
    )
)

print("\n[상위 그룹 간 오분류 패턴]")

print(
    super_error_pairs.to_string(index=False)
)



[상위 그룹 간 오분류 패턴]
          true_super_name           pred_super_name  confusion_count
       GI_Digestive_Group Female_Reproductive_Group               42
Female_Reproductive_Group        GI_Digestive_Group               34
Female_Reproductive_Group      Endocrine_Rare_Group               25
               Lung_Group        GI_Digestive_Group               23
       Skin_Sarcoma_Group Female_Reproductive_Group               21
 Male_Genitourinary_Group Female_Reproductive_Group               20
 Male_Genitourinary_Group        GI_Digestive_Group               19
Female_Reproductive_Group  Male_Genitourinary_Group               17
 Male_Genitourinary_Group      Endocrine_Rare_Group               16
          Head_Neck_Group Female_Reproductive_Group               15
     Endocrine_Rare_Group Female_Reproductive_Group               14
               Lung_Group Female_Reproductive_Group               13
          CNS_Brain_Group Female_Reproductive_Group               13
             Kid

In [ ]:
# =============================================================
# [Step 6-9] 결과 저장
# =============================================================

val_results_df.to_csv(
    "validation_predictions.csv",
    index=False
)

subclass_error_analysis.to_csv(
    "subclass_error_analysis.csv"
)

confusion_pairs.to_csv(
    "subclass_confusion_pairs.csv",
    index=False
)

super_error_pairs.to_csv(
    "superclass_confusion_pairs.csv",
    index=False
)

print("\nValidation 분석 결과 저장 완료!")


Validation 분석 결과 저장 완료!


In [19]:
# =============================================================
# [Step 7-1] 예측 함수 및 Soft Hierarchical 성능 비교 (Validation)
# =============================================================
import torch.nn.functional as F

@torch.no_grad()
def predict_proba(model, features, batch_size=256):
    model.eval()
    sub_list, super_list, sub_final_list = [], [], []
    for i in range(0, len(features), batch_size):
        xb = torch.tensor(features[i:i + batch_size], dtype=torch.float32).to(device)
        # 💡 수정: 모델이 반환하는 3가지 출력값 수신
        super_logits, sub_logits, P_sub_final = model(xb)
        sub_list.append(F.softmax(sub_logits, dim=1).cpu().numpy())
        super_list.append(F.softmax(super_logits, dim=1).cpu().numpy())
        sub_final_list.append(P_sub_final.cpu().numpy())
    return np.vstack(sub_list), np.vstack(super_list), np.vstack(sub_final_list)


# 검증 데이터 예측 확률 추출
P_sub_val, P_super_val, P_sub_final_val = predict_proba(model, X_val_gene)
y_val = np.asarray(y_sub_val)

# 비교 평가
for name, scores in [
    ("Direct Sub Head",    P_sub_val),
    ("Soft Hierarchical",  P_sub_final_val),  # 모델 내부에서 계층 확률이 결합된 최종 출력
]:
    pred = scores.argmax(1)
    print(f"{name:18s} Acc {accuracy_score(y_val, pred):.4f} | "
        f"Macro F1 {f1_score(y_val, pred, average='macro', zero_division=0):.4f}")

Direct Sub Head    Acc 0.3344 | Macro F1 0.3117
Soft Hierarchical  Acc 0.3320 | Macro F1 0.3054


In [ ]:
# =============================================================
# [Step 7-2] Test 예측 및 제출 파일 저장 (Soft Hierarchical 반영)
# =============================================================

# 입력 정합성 확인
assert X_test_gene.shape[1] == input_dim, "test 입력 차원이 학습과 다름"
assert len(test) == X_test_gene.shape[0]
assert test["ID"].is_unique

# 💡 1. sub_label -> super_label 매핑 변수(sub_to_super) 정의 보장
sub_to_super = (
    train.groupby("sub_label")["super_label"].first()
    .reindex(range(num_sub_classes)).to_numpy()
)

# 💡 2. predict_proba 함수 정의
@torch.no_grad()
def predict_proba_test(model, features, batch_size=256):
    model.eval()
    sub_list, super_list, sub_final_list = [], [], []
    for i in range(0, len(features), batch_size):
        xb = torch.tensor(features[i:i + batch_size], dtype=torch.float32).to(device)
        super_logits, sub_logits, P_sub_final = model(xb)
        sub_list.append(F.softmax(sub_logits, dim=1).cpu().numpy())
        super_list.append(F.softmax(super_logits, dim=1).cpu().numpy())
        sub_final_list.append(P_sub_final.cpu().numpy())
    return np.vstack(sub_list), np.vstack(super_list), np.vstack(sub_final_list)

# Test 예측 확률 추출
P_sub_test, P_super_test, P_sub_final_test = predict_proba_test(model, X_test_gene)

# DECODE 설정에 따른 점수 선택 ("sub" 또는 "hier" 계층 통합 확률)
DECODE = "hier"  # 계층 구조가 반영된 P_sub_final_test 활용
scores_test = P_sub_final_test if DECODE == "hier" else P_sub_test

pred_sub_test = scores_test.argmax(1)
pred_super_test = P_super_test.argmax(1)

pred_sub_names = sub_encoder.inverse_transform(pred_sub_test)
pred_super_names = super_encoder.inverse_transform(pred_super_test)

# 제출 파일 (ID, SUBCLASS)
submission_nn = pd.DataFrame({"ID": test["ID"].values, "SUBCLASS": pred_sub_names})
assert submission_nn["SUBCLASS"].notna().all()
submission_nn.to_csv("submission_nn.csv", index=False)

# 분석용 상세 파일 (예측 암종, 상위 그룹, 확신도)
detail_df = pd.DataFrame({
    "ID": test["ID"].values,
    "pred_sub_name": pred_sub_names,
    "pred_super_name": pred_super_names,
    "max_prob": scores_test.max(1) / scores_test.sum(1),
})
detail_df.to_csv("test_predictions_detail.csv", index=False)
np.save("P_sub_test_nn.npy", P_sub_test)
np.save("P_super_test_nn.npy", P_super_test)

# 점검 1: 예측 비율을 train 비율과 비교
check = pd.DataFrame({
    "train(%)": train["SUBCLASS"].value_counts(normalize=True) * 100,
    "test예측(%)": submission_nn["SUBCLASS"].value_counts(normalize=True) * 100,
}).round(1).fillna(0)
check["배수"] = (check["test예측(%)"] / check["train(%)"]).round(2)
print(check.sort_values("배수"))

# 점검 2: 세부 암종 예측이 상위 그룹 예측과 일치하는 비율
consistent = (sub_to_super[pred_sub_test] == pred_super_test).mean()
print(f"\nsub 예측과 super 예측이 일치하는 비율: {consistent:.1%}")
print(submission_nn.head(), submission_nn.shape)

          train(%)  test예측(%)    배수
SUBCLASS                           
GBMLGG         7.4        2.7  0.36
PCPG           2.4        1.0  0.42
SARC           3.2        1.4  0.44
PRAD           4.3        2.0  0.47
HNSC           3.6        1.8  0.50
KIPAN          8.3        4.6  0.55
LAML           2.5        1.4  0.56
KIRC           5.4        3.5  0.65
BRCA          12.7        8.3  0.65
LUSC           2.9        2.1  0.72
LUAD           3.0        2.2  0.73
OV             4.1        3.1  0.76
DLBC           0.6        0.5  0.83
LIHC           2.5        2.1  0.84
THCA           5.2        4.4  0.85
SKCM           4.5        4.1  0.91
LGG            3.7        4.0  1.08
TGCT           2.0        2.4  1.20
PAAD           1.9        2.3  1.21
BLCA           1.7        2.4  1.41
ACC            1.2        1.7  1.42
UCEC           3.2        4.8  1.50
CESC           2.5        4.2  1.68
STES           6.1       12.2  2.00
THYM           1.6        3.2  2.00
COAD           3.6       17.